# Kaggle Qwen2.5 3B GRPO Run

Use this notebook from the Kaggle-backed VS Code kernel. It runs the complementary 3B Unsloth GRPO experiment while the campus server runs the default 1.5B job.

## 1. GPU Check

Expected on Kaggle: two T4 GPUs. If this cell does not show T4s, the notebook is not attached to the Kaggle kernel.

In [1]:
!nvidia-smi

Sat Apr 25 21:55:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Bootstrap Kaggle Working Directory And Repo

The Kaggle kernel cannot see your local laptop path. This section resets the remote kernel to `/kaggle/working`, then clones or updates the pushed `round2-redshift` branch. Run these cells even if a previous `%cd` failed.


In [2]:
import os
from pathlib import Path

Path("/kaggle/working").mkdir(parents=True, exist_ok=True)
os.chdir("/kaggle/working")
print("cwd:", os.getcwd())


cwd: /kaggle/working


In [3]:
import os, shutil, subprocess, time
from pathlib import Path

REPO_URL = "https://github.com/srimanreddy4/MetaHackathon-R2"
BRANCH = "accel-attack"
WORKDIR = Path("/kaggle/working/MetaHackathon-R2")

os.chdir("/kaggle/working")
if (WORKDIR / ".git").exists():
    os.chdir(WORKDIR)
    subprocess.run(["git", "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "checkout", BRANCH], check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    if WORKDIR.exists():
        backup = WORKDIR.with_name(f"{WORKDIR.name}.bak.{int(time.time())}")
        shutil.move(str(WORKDIR), str(backup))
        print("Moved non-git existing directory to", backup)
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(WORKDIR)], check=True)
    os.chdir(WORKDIR)

print("cwd:", os.getcwd())
subprocess.run(["git", "log", "--oneline", "-5"], check=True)


Cloning into '/kaggle/working/MetaHackathon-R2'...


cwd: /kaggle/working/MetaHackathon-R2
9c8272c pull and fix
2bb62a1 Merge branch 'round2-redshift' of https://github.com/srimanreddy4/MetaHackathon-R2 into accel-attack
7fc73d4 .
34016dc non verbose
4735ce8 update mutator algo


CompletedProcess(args=['git', 'log', '--oneline', '-5'], returncode=0)

In [4]:
%cd /kaggle/working/MetaHackathon-R2
%env PYTHONPATH=src:scripts
!python - <<'PY'
import os
from pathlib import Path
print("cwd=", Path.cwd())
print("PYTHONPATH=", os.environ.get("PYTHONPATH"))
print("repo exists=", Path("src/oncallenv").exists())



/kaggle/working
env: PYTHONPATH=src:scripts
/bin/bash: line 1: warning: here-document at line 1 delimited by end-of-file (wanted `PY')
cwd= /kaggle/working/MetaHackathon-R2
PYTHONPATH= src:scripts
repo exists= True


## 3. Install Dependencies

Run the normal install first. If Kaggle has a PyTorch/CUDA dependency conflict, use the fallback cell after it.

In [5]:
!python -m pip install -U pip setuptools wheel
!python -m pip install -r requirements.txt
!python -m pip install -r requirements-llm.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 31.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 58.3 MB/s eta 0:00:00
  Attempting uninstall: wheel
    Found existing installation: wheel 0.46.3
    Uninstalling wheel-0.46.3:
      Successfully uninstalled wheel-0.46.3
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.6/728.6 kB 26.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16/16 [openenv-core] [openenv-core]tic]
INFO: pip is looking at multiple versions of unsloth to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of unsloth to dete

In [44]:
# Fallback install if the previous cell fails:
# !python -m pip install -U transformers datasets accelerate trl peft bitsandbytes unsloth

## 4. Verify Environment

Expected: `21 passed` and OpenEnv validation OK.

In [6]:
!bash scripts/run_kaggle_qwen3b_grpo.sh verify

........................                                                 [100%]
=============================== warnings summary ===============================
../../../usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64
  /usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
    prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))

../../../usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85
../../../usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85
../../../usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85
../../../usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85
../../../usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85
../../../usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85
  /usr/local/l

## 5. Small 3B Smoke Run

This is intentionally small. Continue only if it writes `training_results/unsloth_grpo_qwen3b_smoke/summary.json`.

In [ ]:
!bash scripts/run_kaggle_qwen3b_grpo.sh smoke

In [ ]:
!cat training_results/unsloth_grpo_qwen3b_smoke/summary.json

## 6. Main Kaggle 3B Run

This is the main complementary run: Qwen2.5 3B, 160 curriculum tasks, 600 GRPO steps.

In [7]:
!bash scripts/run_kaggle_qwen3b_grpo.sh easy-main

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
2026-04-25 21:56:49.643055: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777154209.833484     168 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777154209.889480     168 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777154210.317892     168 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777154210.317949     168 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:17

In [ ]:
!cat training_results/unsloth_grpo_qwen3b_kaggle/summary.json
!bash scripts/run_kaggle_qwen3b_grpo.sh summary

In [14]:
!ls -lh training_results/unsloth_grpo_qwen3b_kaggle
!ls -lh training_results/unsloth_grpo_qwen3b_kaggle/checkpoint-300
!cat training_results/unsloth_grpo_qwen3b_kaggle/checkpoint-300/trainer_state.json | tail -80


total 256K
-rw-r--r-- 1 root root  27K Apr 25 03:25 baseline_generations.json
drwxr-xr-x 2 root root 4.0K Apr 25 04:51 checkpoint-100
drwxr-xr-x 2 root root 4.0K Apr 25 06:17 checkpoint-200
drwxr-xr-x 2 root root 4.0K Apr 25 07:43 checkpoint-300
-rw-r--r-- 1 root root 211K Apr 25 03:20 dataset.jsonl
-rw-r--r-- 1 root root 2.1K Apr 25 07:43 README.md
total 189M
-rw-r--r-- 1 root root 1.2K Apr 25 07:43 adapter_config.json
-rw-r--r-- 1 root root 115M Apr 25 07:43 adapter_model.safetensors
-rw-r--r-- 1 root root  605 Apr 25 07:43 added_tokens.json
-rw-r--r-- 1 root root 2.5K Apr 25 07:43 chat_template.jinja
-rw-r--r-- 1 root root 1.6M Apr 25 07:43 merges.txt
-rw-r--r-- 1 root root  59M Apr 25 07:43 optimizer.pt
-rw-r--r-- 1 root root 5.2K Apr 25 07:43 README.md
-rw-r--r-- 1 root root  15K Apr 25 07:43 rng_state.pth
-rw-r--r-- 1 root root 1.4K Apr 25 07:43 scaler.pt
-rw-r--r-- 1 root root 1.5K Apr 25 07:43 scheduler.pt
-rw-r--r-- 1 root root  614 Apr 25 07:43 special_tokens_map.json
-rw-r--

## 7. Optional Extended Run

Run this only if the main 600-step job is stable and Kaggle time remains.

In [ ]:
# !bash scripts/run_kaggle_qwen3b_grpo.sh long

## 8. OOM Fallback

If the 3B smoke run OOMs, run this 1.5B ablation instead. It still gives a Kaggle portability result.

In [ ]:
# !bash scripts/run_kaggle_qwen3b_grpo.sh fallback-1b5

## 9. Archive Outputs

Download `/kaggle/working/qwen3b_grpo_results.tar.gz` from Kaggle outputs.

In [ ]:
!bash scripts/run_kaggle_qwen3b_grpo.sh archive